# 08 – Interoperability

> Runs against the catalog from `CATALOG` (Lakekeeper, Polaris, RustFS or Unity; default Lakekeeper). What a catalog cannot do is shown by the cell instead of aborting.

A table written by one engine, read by all others – through the same
catalog (warehouse with vended or own credentials). Uses the adapters from `iceberg_lab.engines` that the tests use too.
Engines that are not running (Trino, pg_lake, Spark without Java) are skipped.

In [ ]:
from iceberg_lab.config import (
    PROFILE,
    SETTINGS,
    WAREHOUSE_STS,
)
from iceberg_lab.notebook import header

CATALOG_URI = SETTINGS.catalog_uri
S3_ENDPOINT = SETTINGS.s3_endpoint
header()

In [ ]:
import uuid

import pandas as pd
import pyarrow as pa

from iceberg_lab.engines import (
    SCHEMA,
    DaftEngine,
    DuckDBEngine,
    PgLakeEngine,
    PolarsEngine,
    PyIcebergEngine,
    SparkEngine,
    TrinoEngine,
    pyiceberg_catalog,
    spark_session,
)

engines = {
    "pyiceberg": PyIcebergEngine(WAREHOUSE_STS),
    "polars": PolarsEngine(WAREHOUSE_STS),
    "daft": DaftEngine(WAREHOUSE_STS),
}
for name, factory in {
    "duckdb": lambda: DuckDBEngine(WAREHOUSE_STS),
    "spark": lambda: SparkEngine(spark_session(), WAREHOUSE_STS),
    "trino": lambda: TrinoEngine(WAREHOUSE_STS),
    "pg_lake": lambda: PgLakeEngine(WAREHOUSE_STS),
}.items():
    if name == "pg_lake" and PROFILE.auth == "sigv4":
        print(
            "pg_lake skipped: pg_lake has no SigV4 (rest_auth_type: default, oauth2, horizon, none)"
        )
        continue
    try:
        e = factory()
        e.read  # noqa: B018
        engines[name] = e
    except Exception as exc:  # noqa: BLE001 - the matrix records every failure
        print(f"{name} skipped: {exc}")

NS = "nb_interop"
pyiceberg_catalog(WAREHOUSE_STS).create_namespace_if_not_exists(NS)
data = pa.table(
    {"id": [1, 2, 3], "name": ["a", "b", "c"], "amount": [1.5, 2.5, 3.5]}, schema=SCHEMA
)
list(engines)

In [ ]:
matrix = {}
for writer, w in engines.items():
    table = f"from_{writer}_{uuid.uuid4().hex[:6]}"
    try:
        w.write(NS, table, data)
    except Exception as exc:  # noqa: BLE001 - the matrix records every failure
        matrix[writer] = {r: f"write failed: {type(exc).__name__}" for r in engines}
        continue
    row = {}
    for reader, r in engines.items():
        try:
            row[reader] = (
                "✅"
                if r.read(NS, table) == sorted(zip(*data.to_pydict().values()))
                else "❌ falsch"
            )
        except Exception as exc:  # noqa: BLE001 - the matrix records every failure
            row[reader] = f"❌ {type(exc).__name__}"
    matrix[writer] = row

pd.DataFrame(matrix).T.rename_axis(index="writer ↓ / reader →")